# SurfaceTarget-CRC — phase 01 — patch 1 15

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PATCH 1.15
#
# PROVISIONAL RNA-SEQ FILE SELECTION
# AND DOWNLOAD MANIFEST GENERATION
#
# Objectives:
# 1. Select one primary tumor RNA-seq file per patient
# 2. Apply documented decisions for ambiguous patients
# 3. Preserve alternative files for sensitivity analysis
# 4. Validate sample identifiers and cohort membership
# 5. Generate reproducible download manifests
# 6. Examine selected workflow distributions
# 7. Create a research checkpoint
#
# IMPORTANT:
# Selection is provisional pending full-cohort QC.
# No additional molecular files are downloaded.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import json

# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"

RESULTS = ROOT / "results/tables"

CHECKPOINTS = ROOT / "checkpoints"

RESULTS.mkdir(parents=True, exist_ok=True)
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

WORKFLOW_PATH = (
    META / "tcga_cohort_workflow_metadata.csv"
)

COHORT_PATH = (
    META / "tcga_mss_evidence_tiers.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.15 — RNA-SEQ FILE SELECTION")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load metadata
# ------------------------------------------------------------

for path in [WORKFLOW_PATH, COHORT_PATH]:

    if not path.exists():
        raise FileNotFoundError(
            f"Required metadata missing: {path}"
        )

inventory = pd.read_csv(
    WORKFLOW_PATH,
    dtype=str,
    keep_default_na=False
)

cohort = pd.read_csv(
    COHORT_PATH,
    dtype=str,
    keep_default_na=False
)

print("\nMetadata loaded successfully.")

print(
    "Total files in workflow inventory:",
    len(inventory)
)

print(
    "Total cohort records:",
    len(cohort)
)

# ------------------------------------------------------------
# 3. Separate tumor and normal tissue
# ------------------------------------------------------------

tumor = inventory[
    inventory["sample_type"] == "Primary Tumor"
].copy()

normal = inventory[
    inventory["sample_type"] == "Solid Tissue Normal"
].copy()

print("\nCandidate tumor files:", len(tumor))

print("Normal tissue files:", len(normal))

print(
    "Unique tumor patients:",
    tumor["patient_id"].nunique()
)

# ------------------------------------------------------------
# 4. Validate source inventory
# ------------------------------------------------------------

if inventory["file_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate file IDs in source inventory."
    )

if tumor["patient_id"].nunique() != 421:
    raise RuntimeError(
        "Unexpected tumor patient count."
    )

if len(tumor) != 425:
    raise RuntimeError(
        "Unexpected candidate tumor file count."
    )

if len(normal) != 51:
    raise RuntimeError(
        "Unexpected normal tissue file count."
    )

if not tumor["cohort_tier"].isin(
    ["Tier A", "Tier B"]
).all():
    raise RuntimeError(
        "Unexpected tumor cohort tier."
    )

print("\nSource inventory validation: PASSED")

# ------------------------------------------------------------
# 5. Define documented selections
# ------------------------------------------------------------

# These decisions are based on the detailed
# biospecimen and RNA-seq QC investigations
# completed in Patches 1.11 through 1.14.

MANUAL_SELECTIONS = {

    "TCGA-A6-2677": {

        "file_id":
        "c834cb75-259c-40bd-93cc-48f9756cc47a",

        "reason": (
            "Selected 01A aliquot. "
            "High gene-assignment fraction, "
            "consistent protein-coding detection, "
            "and strong correlation with the "
            "alternative 01A profile. "
            "Workflow version represented in "
            "the wider cohort."
        )
    },

    "TCGA-A6-3810": {

        "file_id":
        "cd10bd5e-b75b-4d47-a103-9c259dcf1a19",

        "reason": (
            "Selected 01A aliquot using the "
            "most common recorded workflow version. "
            "Higher assigned gene counts and "
            "protein-coding detection at CPM >= 1 "
            "than the alternative 01A profile. "
            "Strong within-vial expression correlation."
        )
    }
}

# ------------------------------------------------------------
# 6. Identify ambiguous patients
# ------------------------------------------------------------

patient_file_counts = (
    tumor.groupby("patient_id")["file_id"]
    .nunique()
)

ambiguous_patients = set(
    patient_file_counts[
        patient_file_counts > 1
    ].index
)

print("\n" + "-" * 65)
print("AMBIGUOUS PATIENT VALIDATION")
print("-" * 65)

print(
    "\nPatients with multiple files:",
    len(ambiguous_patients)
)

if ambiguous_patients != set(MANUAL_SELECTIONS):

    raise RuntimeError(
        "Ambiguous patient list differs "
        "from documented selection decisions."
    )

print("\nAmbiguous patient validation: PASSED")

# ------------------------------------------------------------
# 7. Construct selected file list
# ------------------------------------------------------------

selection_records = []

for patient_id, group in tumor.groupby("patient_id"):

    available_ids = set(
        group["file_id"]
    )

    if len(group) == 1:

        selected_id = group.iloc[0]["file_id"]

        selection_type = "Single available file"

        selection_reason = (
            "Only one eligible primary tumor "
            "STAR Counts file available."
        )

    else:

        decision = MANUAL_SELECTIONS[patient_id]

        selected_id = decision["file_id"]

        if selected_id not in available_ids:

            raise RuntimeError(
                f"Selected file unavailable: "
                f"{patient_id}"
            )

        selection_type = "Documented manual selection"

        selection_reason = decision["reason"]

    selection_records.append({

        "patient_id": patient_id,

        "selected_file_id": selected_id,

        "candidate_files": len(group),

        "selection_type": selection_type,

        "selection_reason": selection_reason
    })

selection_table = pd.DataFrame(
    selection_records
)

# ------------------------------------------------------------
# 8. Create selected tumor manifest
# ------------------------------------------------------------

selected = tumor.merge(
    selection_table,
    on="patient_id",
    how="inner",
    validate="many_to_one"
)

selected = selected[
    selected["file_id"]
    == selected["selected_file_id"]
].copy()

selected = selected.sort_values(
    ["project", "cohort_tier", "patient_id"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 9. Validate one file per patient
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SELECTED FILE VALIDATION")
print("-" * 65)

if len(selected) != 421:
    raise RuntimeError(
        "Selected tumor file count is not 421."
    )

if selected["patient_id"].nunique() != 421:
    raise RuntimeError(
        "Some patients have multiple or missing files."
    )

if selected["file_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate selected file IDs detected."
    )

selected_counts = (
    selected.groupby("patient_id")
    .size()
)

if not selected_counts.eq(1).all():
    raise RuntimeError(
        "One-file-per-patient validation failed."
    )

print("\nOne tumor file per patient: PASSED")

# ------------------------------------------------------------
# 10. Validate cohort evidence membership
# ------------------------------------------------------------

eligible_cohort = cohort[
    cohort["evidence_tier"].isin([
        "Tier A - Published MSS and concordant low MSI",
        "Tier B - Concordant low MSI only"
    ])
].copy()

if eligible_cohort["patientId"].duplicated().any():
    raise RuntimeError(
        "Duplicate patient identifiers in evidence tiers."
    )

if set(selected["patient_id"]) != set(
    eligible_cohort["patientId"]
):
    raise RuntimeError(
        "Selected patients differ from eligible cohorts."
    )

# Attach original MSI identifiers to the manifest.

msi_columns = [
    "patientId",
    "mantis_sample_id",
    "msisensor_sample_id",
    "evidence_tier"
]

selected = selected.merge(
    eligible_cohort[msi_columns],
    left_on="patient_id",
    right_on="patientId",
    how="left",
    validate="one_to_one"
)

# TCGA sample-type barcode correspondence.
#
# The first 15 characters identify the
# patient and sample-type code.
#
# This does not prove that the RNA-seq and
# MSI measurements used identical aliquots.

sample_matches = (
    selected["sample_id"].str[:15]
    ==
    selected["mantis_sample_id"].str[:15]
)

if not sample_matches.all():

    mismatches = selected.loc[
        ~sample_matches,
        ["patient_id", "sample_id", "mantis_sample_id"]
    ]

    print(mismatches.to_string(index=False))

    raise RuntimeError(
        "Selected RNA-seq/MSI sample linkage failed."
    )

print("MSI sample-type correspondence: PASSED")

# ------------------------------------------------------------
# 11. Validate cohort-specific counts
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("FINAL SELECTED TUMOR FILE COUNTS")
print("-" * 65)

cohort_counts = (
    selected.groupby(
        ["project", "cohort_tier"]
    )
    .agg(
        patients=("patient_id", "nunique"),
        selected_files=("file_id", "nunique")
    )
    .reset_index()
)

print("\n")
print(cohort_counts.to_string(index=False))

expected_counts = {
    ("TCGA-COAD", "Tier A"): 98,
    ("TCGA-COAD", "Tier B"): 194,
    ("TCGA-READ", "Tier A"): 46,
    ("TCGA-READ", "Tier B"): 83
}

for (project, tier), expected in expected_counts.items():

    actual = len(
        selected[
            (selected["project"] == project)
            & (selected["cohort_tier"] == tier)
        ]
    )

    if actual != expected:

        raise RuntimeError(
            f"Unexpected cohort size for "
            f"{project} {tier}: {actual}"
        )

print("\nCohort count validation: PASSED")

# ------------------------------------------------------------
# 12. Identify excluded alternative files
# ------------------------------------------------------------

selected_ids = set(
    selected["file_id"]
)

excluded = tumor[
    ~tumor["file_id"].isin(selected_ids)
].copy()

excluded["exclusion_reason"] = (
    "Alternative tumor RNA-seq file. "
    "Excluded from primary analysis to prevent "
    "multiple measurements per patient. "
    "Retained for sensitivity analysis."
)

print("\n" + "-" * 65)
print("ALTERNATIVE FILE AUDIT")
print("-" * 65)

print("\nAlternative files excluded:", len(excluded))

if len(excluded) != 4:
    raise RuntimeError(
        "Unexpected number of alternative files."
    )

print("\nExcluded alternatives:")

print(
    excluded[
        [
            "patient_id",
            "sample_id",
            "file_id",
            "version_short"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 13. Preserve normal tissue inventory
# ------------------------------------------------------------

normal = normal.sort_values(
    ["project", "patient_id"]
).reset_index(drop=True)

print("\n" + "-" * 65)
print("NORMAL TISSUE CONTROL INVENTORY")
print("-" * 65)

normal_summary = (
    normal.groupby("project")
    .agg(
        patients=("patient_id", "nunique"),
        files=("file_id", "nunique")
    )
    .reset_index()
)

print("\n")
print(normal_summary.to_string(index=False))

print("\nNormal tissues are not assigned MSS status.")

# ------------------------------------------------------------
# 14. Selected workflow version distribution
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SELECTED WORKFLOW VERSION DISTRIBUTION")
print("-" * 65)

workflow_distribution = pd.crosstab(
    [
        selected["project"],
        selected["cohort_tier"]
    ],
    selected["version_short"]
)

print("\nTumor samples:")

print(workflow_distribution.to_string())

print("\nNormal tissue workflow versions:")

normal_workflows = pd.crosstab(
    normal["project"],
    normal["version_short"]
)

print(normal_workflows.to_string())

# ------------------------------------------------------------
# 15. Storage requirements
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SELECTED DOWNLOAD STORAGE")
print("-" * 65)

selected["file_size_bytes"] = pd.to_numeric(
    selected["file_size_bytes"],
    errors="raise"
)

normal["file_size_bytes"] = pd.to_numeric(
    normal["file_size_bytes"],
    errors="raise"
)

tumor_gb = (
    selected["file_size_bytes"].sum()
    / (1024 ** 3)
)

normal_gb = (
    normal["file_size_bytes"].sum()
    / (1024 ** 3)
)

print(f"\nSelected tumor files: {tumor_gb:.3f} GiB")

print(f"Normal tissue files: {normal_gb:.3f} GiB")

print(
    f"Combined storage: {tumor_gb + normal_gb:.3f} GiB"
)

# ------------------------------------------------------------
# 16. Prepare local download paths
# ------------------------------------------------------------

# These paths will be used by the downloader
# in the next patch.

TUMOR_RAW_DIR = (
    ROOT / "data/raw/tcga_star_counts/tumor"
)

NORMAL_RAW_DIR = (
    ROOT / "data/raw/tcga_star_counts/normal"
)

selected["local_path"] = selected["file_id"].apply(
    lambda fid: str(
        TUMOR_RAW_DIR / f"{fid}.tsv"
    )
)

normal["local_path"] = normal["file_id"].apply(
    lambda fid: str(
        NORMAL_RAW_DIR / f"{fid}.tsv"
    )
)

selected["selection_status"] = (
    "Provisional - pending full cohort QC"
)

normal["selection_status"] = (
    "Candidate normal control - pending QC"
)

# ------------------------------------------------------------
# 17. Save final manifests
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RNA-SEQ MANIFESTS")
print("-" * 65)

tumor_manifest_path = (
    META / "tcga_selected_tumor_rnaseq_manifest.csv"
)

normal_manifest_path = (
    META / "tcga_normal_rnaseq_manifest.csv"
)

excluded_path = (
    META / "tcga_excluded_alternative_rnaseq_files.csv"
)

selection_path = (
    META / "tcga_rnaseq_selection_decisions.csv"
)

workflow_path = (
    RESULTS / "tcga_selected_workflow_distribution.csv"
)

selected.to_csv(
    tumor_manifest_path,
    index=False
)

normal.to_csv(
    normal_manifest_path,
    index=False
)

excluded.to_csv(
    excluded_path,
    index=False
)

selection_table.to_csv(
    selection_path,
    index=False
)

workflow_distribution.to_csv(
    workflow_path
)

print("\nAll manifests saved successfully.")

# ------------------------------------------------------------
# 18. Save selection methodology
# ------------------------------------------------------------

methodology = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.15",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "selection_policy_version": "1.15-v1",

    "selection_status": (
        "Provisional pending full cohort QC"
    ),

    "primary_rule": (
        "Retain one STAR Counts primary tumor "
        "expression file per patient."
    ),

    "ambiguous_patient_decisions": MANUAL_SELECTIONS,

    "selected_tumor_patients": int(
        selected["patient_id"].nunique()
    ),

    "selected_tumor_files": int(
        selected["file_id"].nunique()
    ),

    "candidate_normal_files": int(
        normal["file_id"].nunique()
    ),

    "excluded_alternative_files": int(
        len(excluded)
    ),

    "count_column": "unstranded",

    "download_performed": False,

    "limitations": [
        "File selection is provisional.",
        "Full-cohort RNA-seq QC remains pending.",
        "Normal samples are not automatically matched controls.",
        "Tier B is not categorically verified MSS.",
        "Alternative aliquots should be examined in sensitivity analyses.",
        "Workflow-version differences require downstream assessment.",
        "MSI and RNA-seq matching does not establish identical aliquots."
    ]
}

methodology_path = (
    META / "tcga_rnaseq_final_selection_methodology.json"
)

with open(methodology_path, "w") as f:

    json.dump(
        methodology,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 19. Create research checkpoint
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING RESEARCH CHECKPOINT")
print("-" * 65)

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

checkpoint_path = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_Patch115_{timestamp}.zip"
)

backup_folders = [
    "data/metadata",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]

with ZipFile(
    checkpoint_path,
    "w",
    ZIP_DEFLATED
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if folder_path.exists():

            for file_path in folder_path.rglob("*"):

                if file_path.is_file():

                    archive.write(
                        file_path,
                        arcname=str(
                            file_path.relative_to(ROOT)
                        )
                    )

checkpoint_size_mb = (
    checkpoint_path.stat().st_size
    / (1024 ** 2)
)

print("\nCheckpoint created successfully.")

print(
    f"Checkpoint size: {checkpoint_size_mb:.2f} MB"
)

print("\nCheckpoint location:")
print(checkpoint_path)

# ------------------------------------------------------------
# 20. Final project summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.15 SUMMARY")
print("=" * 65)

print("\nTumor patients:", len(selected))

print(
    "Selected tumor RNA-seq files:",
    selected["file_id"].nunique()
)

print(
    "Normal tissue files:",
    normal["file_id"].nunique()
)

print(
    "Excluded alternative files:",
    len(excluded)
)

print("\nFiles saved:")

print("1. tcga_selected_tumor_rnaseq_manifest.csv")

print("2. tcga_normal_rnaseq_manifest.csv")

print("3. tcga_excluded_alternative_rnaseq_files.csv")

print("4. tcga_rnaseq_selection_decisions.csv")

print("5. tcga_selected_workflow_distribution.csv")

print("6. tcga_rnaseq_final_selection_methodology.json")

print("\nResearch checkpoint: CREATED")

print("\nNo additional molecular files downloaded.")

print("\nRNA-seq manifests are ready for download.")

print("\nPATCH 1.15 COMPLETE")

print("=" * 65)